# Hoja de trabajo 2 — Transformers y mecanismos de atención
CC3092 Deep Learning y Sistemas Inteligentes.

Contenido: (A) ejercicio a mano y atención implementada desde cero, (B) verificaciones numéricas (escalamiento por √d_k, multi-head, máscara causal, costo computacional) y (C) visualización de atención en BERT y GPT-2.

In [1]:
import math, time, os
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModel
torch.manual_seed(0); np.random.seed(0)
os.makedirs("figs", exist_ok=True)

C:\Users\aeeh2\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## A. Ejercicio a mano: atención de producto punto escalado
Secuencia de 3 tokens, d_k = d_v = 2. Se calcula `softmax(QKᵀ/√d_k)V` paso a paso y se compara con la implementación desde cero y con `F.scaled_dot_product_attention`.

In [2]:
Q = torch.tensor([[1.,0.],[0.,1.],[1.,1.]])
K = Q.clone()
V = torch.tensor([[1.,2.],[3.,4.],[5.,6.]])
dk = Q.shape[-1]

def attention(Q, K, V, mask=None):
    scores = Q @ K.transpose(-2, -1) / math.sqrt(Q.shape[-1])
    if mask is not None:
        scores = scores.masked_fill(~mask, float("-inf"))
    w = torch.softmax(scores, dim=-1)
    return w @ V, w, scores

out, W, S = attention(Q, K, V)
print("QK^T =\n", Q @ K.T)
print("QK^T/sqrt(dk) =\n", S)
print("Pesos softmax =\n", W)
print("Salida =\n", out)
ref = F.scaled_dot_product_attention(Q[None], K[None], V[None])[0]
print("Coincide con F.scaled_dot_product_attention:", torch.allclose(out, ref, atol=1e-6))

QK^T =
 tensor([[1., 0., 1.],
        [0., 1., 1.],
        [1., 1., 2.]])
QK^T/sqrt(dk) =
 tensor([[0.7071, 0.0000, 0.7071],
        [0.0000, 0.7071, 0.7071],
        [0.7071, 0.7071, 1.4142]])
Pesos softmax =
 tensor([[0.4011, 0.1978, 0.4011],
        [0.1978, 0.4011, 0.4011],
        [0.2483, 0.2483, 0.5035]])
Salida =
 tensor([[3.0000, 4.0000],
        [3.4067, 4.4067],
        [3.5105, 4.5105]])
Coincide con F.scaled_dot_product_attention: True


## B. Verificaciones (sección 6.1)
### B.1 Varianza de q·k y efecto del escalamiento

In [3]:
dks = [4, 16, 64, 256, 1024]
N = 20000
var_raw, var_scaled = [], []
for d in dks:
    q = torch.randn(N, d); k = torch.randn(N, d)
    dot = (q * k).sum(-1)
    var_raw.append(dot.var().item()); var_scaled.append((dot / math.sqrt(d)).var().item())
    print(f"d_k={d:5d}  var(q.k)={var_raw[-1]:9.2f}  var(q.k/sqrt(dk))={var_scaled[-1]:.3f}")

# Efecto en softmax y gradiente (n=16 llaves)
n = 16
max_p_raw, max_p_sc, g_raw, g_sc = [], [], [], []
for d in dks:
    gr, gs, pr, ps = [], [], [], []
    for _ in range(200):
        q = torch.randn(d); K_ = torch.randn(n, d)
        for scale, lst_g, lst_p in ((1.0, gr, pr), (1/math.sqrt(d), gs, ps)):
            s = (K_ @ q * scale).clone().requires_grad_(True)
            p = torch.softmax(s, 0)
            lst_p.append(p.max().item())
            p[0].backward()
            lst_g.append(s.grad.norm().item())
    max_p_raw.append(np.mean(pr)); max_p_sc.append(np.mean(ps)); g_raw.append(np.mean(gr)); g_sc.append(np.mean(gs))
    print(f"d_k={d:5d}  max p: sin escalar={max_p_raw[-1]:.3f} escalado={max_p_sc[-1]:.3f} | |grad|: sin escalar={g_raw[-1]:.2e} escalado={g_sc[-1]:.2e}")

fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
ax[0].loglog(dks, var_raw, "o-", label="sin escalar"); ax[0].loglog(dks, var_scaled, "s-", label="÷√d_k")
ax[0].loglog(dks, dks, "k--", lw=.8, label="teoría: d_k"); ax[0].set_xlabel("d_k"); ax[0].set_ylabel("Var(q·k)"); ax[0].legend(); ax[0].set_title("Varianza del producto punto")
ax[1].semilogx(dks, max_p_raw, "o-", label="sin escalar"); ax[1].semilogx(dks, max_p_sc, "s-", label="÷√d_k")
ax[1].set_xlabel("d_k"); ax[1].set_ylabel("máx. probabilidad softmax"); ax[1].legend(); ax[1].set_title("Saturación del softmax (n=16)")
ax[2].loglog(dks, g_raw, "o-", label="sin escalar"); ax[2].loglog(dks, g_sc, "s-", label="÷√d_k")
ax[2].set_xlabel("d_k"); ax[2].set_ylabel("‖∂p₀/∂s‖"); ax[2].legend(); ax[2].set_title("Gradiente a través del softmax")
plt.tight_layout(); plt.savefig("figs/escalamiento.png", dpi=170); plt.show()

d_k=    4  var(q.k)=     3.98  var(q.k/sqrt(dk))=0.994
d_k=   16  var(q.k)=    16.07  var(q.k/sqrt(dk))=1.004
d_k=   64  var(q.k)=    64.25  var(q.k/sqrt(dk))=1.004
d_k=  256  var(q.k)=   253.06  var(q.k/sqrt(dk))=0.989


d_k= 1024  var(q.k)=  1015.56  var(q.k/sqrt(dk))=0.992
d_k=    4  max p: sin escalar=0.428 escalado=0.230 | |grad|: sin escalar=4.90e-02 escalado=5.60e-02
d_k=   16  max p: sin escalar=0.660 escalado=0.231 | |grad|: sin escalar=4.07e-02 escalado=5.77e-02
d_k=   64  max p: sin escalar=0.834 escalado=0.245 | |grad|: sin escalar=1.91e-02 escalado=5.77e-02


d_k=  256  max p: sin escalar=0.910 escalado=0.242 | |grad|: sin escalar=1.19e-02 escalado=5.90e-02
d_k= 1024  max p: sin escalar=0.963 escalado=0.242 | |grad|: sin escalar=5.09e-03 escalado=5.76e-02


C:\Users\aeeh2\AppData\Local\Temp\ipykernel_14364\1213615682.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig("figs/escalamiento.png", dpi=170); plt.show()


### B.2 Multi-head: número de parámetros

In [4]:
d_model = 512
for h in (1, 8):
    mha = nn.MultiheadAttention(d_model, h, batch_first=True)
    print(f"h={h}: parámetros = {sum(p.numel() for p in mha.parameters()):,}  (d_head={d_model//h})")

h=1: parámetros = 1,050,624  (d_head=512)
h=8: parámetros = 1,050,624  (d_head=64)


### B.3 Máscara causal

In [5]:
L = 5
mask = nn.Transformer.generate_square_subsequent_mask(L)
print(mask)
x = torch.randn(1, L, 16)
mha = nn.MultiheadAttention(16, 4, batch_first=True)
_, w = mha(x, x, x, attn_mask=mask, need_weights=True)
print("¿Triangular inferior?", torch.allclose(w[0], torch.tril(w[0])))
print("Filas suman 1:", torch.allclose(w[0].sum(-1), torch.ones(L)))

tensor([[0., -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0.]])
¿Triangular inferior? True
Filas suman 1: True


### B.4 Costo computacional: tiempo y memoria vs. longitud n
Se mide la atención ingenua (materializa la matriz n×n) y `F.scaled_dot_product_attention`. La memoria de la matriz de puntajes se calcula analíticamente (4·n² bytes por cabeza en float32).

In [6]:
ns = [128, 256, 512, 1024, 2048, 4096]
d = 64
t_naive, t_sdpa = [], []
def timeit(fn, reps=3):
    fn(); best = 1e9
    for _ in range(reps):
        t = time.perf_counter(); fn(); best = min(best, time.perf_counter() - t)
    return best
for n in ns:
    q = torch.randn(1, 8, n, d); k = torch.randn(1, 8, n, d); v = torch.randn(1, 8, n, d)
    t_naive.append(timeit(lambda: torch.softmax(q @ k.transpose(-2, -1) / math.sqrt(d), -1) @ v))
    t_sdpa.append(timeit(lambda: F.scaled_dot_product_attention(q, k, v)))
    print(f"n={n:5d} naive={t_naive[-1]*1e3:8.1f} ms  sdpa={t_sdpa[-1]*1e3:8.1f} ms")
mem_mb = [8 * 4 * n * n / 2**20 for n in ns]   # 8 cabezas

fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
ax[0].loglog(ns, t_naive, "o-", label="ingenua"); ax[0].loglog(ns, t_sdpa, "s-", label="F.sdpa")
ref = [t_naive[0] * (n / ns[0]) ** 2 for n in ns]; ax[0].loglog(ns, ref, "k--", lw=.8, label="∝ n²")
ax[0].set_xlabel("n"); ax[0].set_ylabel("tiempo (s)"); ax[0].legend(); ax[0].set_title("Tiempo (CPU, 8 cabezas, d=64)")
ax[1].loglog(ns, mem_mb, "o-"); ax[1].set_xlabel("n"); ax[1].set_ylabel("MB"); ax[1].set_title("Memoria matriz de atención (8 cabezas)")
plt.tight_layout(); plt.savefig("figs/costo.png", dpi=170); plt.show()

n=  128 naive=     0.3 ms  sdpa=     0.2 ms
n=  256 naive=     1.0 ms  sdpa=     0.7 ms
n=  512 naive=     4.4 ms  sdpa=     2.5 ms


n= 1024 naive=    21.8 ms  sdpa=    10.9 ms


n= 2048 naive=    78.6 ms  sdpa=    23.4 ms


n= 4096 naive=   357.0 ms  sdpa=   134.6 ms


C:\Users\aeeh2\AppData\Local\Temp\ipykernel_14364\1434727469.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig("figs/costo.png", dpi=170); plt.show()


## C. Atención en BERT y GPT-2

In [7]:
tok_bert = AutoTokenizer.from_pretrained("bert-base-uncased")
bert = AutoModel.from_pretrained("bert-base-uncased", output_attentions=True, attn_implementation="eager").eval()
tok_gpt2 = AutoTokenizer.from_pretrained("gpt2")
gpt2 = AutoModel.from_pretrained("gpt2", output_attentions=True, attn_implementation="eager").eval()

def get_attn(tok, model, text):
    enc = tok(text, return_tensors="pt")
    with torch.no_grad():
        out = model(**enc)
    att = torch.stack(out.attentions).squeeze(1)   # (capas, cabezas, n, n)
    return tok.convert_ids_to_tokens(enc["input_ids"][0]), att

S1 = "The animal didn't cross the street because it was too tired."
S2 = "The animal didn't cross the street because it was too wide."
toks, att = get_attn(tok_bert, bert, S1)
print(toks); print("forma:", tuple(att.shape))

C:\Users\aeeh2\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\aeeh2\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


C:\Users\aeeh2\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\aeeh2\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:  17%|█▋        | 34/199 [00:00<00:00, 318.54it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 978.27it/s]

...

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights:   8%|▊         | 12/148 [00:00<00:01, 113.73it/s]

Loading weights:  16%|█▌        | 24/148 [00:00<00:01, 70.38it/s] 

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 447.04it/s]

['[CLS]', 'the', 'animal', 'didn', "'", 't', 'cross', 'the', 'street', 'because', 'it', 'was', 'too', 'tired', '.', '[SEP]']
forma: (12, 12, 16, 16)


### C.1 Mapas de calor de 4 cabezas de capas distintas (BERT)

In [8]:
def heat(ax, a, toks, title):
    ax.imshow(a, cmap="viridis", vmin=0, vmax=min(1, a.max()))
    ax.set_xticks(range(len(toks))); ax.set_xticklabels(toks, rotation=90, fontsize=7)
    ax.set_yticks(range(len(toks))); ax.set_yticklabels(toks, fontsize=7); ax.set_title(title, fontsize=9)

heads_bert = [(0, 2), (3, 1), (7, 10), (10, 3)]   # (capa, cabeza), 0-indexadas
fig, axs = plt.subplots(1, 4, figsize=(17, 4.6))
for ax, (l, h) in zip(axs, heads_bert):
    heat(ax, att[l, h].numpy(), toks, f"BERT capa {l+1}, cabeza {h+1}")
plt.tight_layout(); plt.savefig("figs/bert_heads.png", dpi=170); plt.show()

C:\Users\aeeh2\AppData\Local\Temp\ipykernel_14364\1757959305.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig("figs/bert_heads.png", dpi=170); plt.show()


### C.2 «it» → «animal» vs. «street» en BERT
Para cada cabeza se mide la atención de «it» hacia «animal» y «street» en ambas oraciones; se busca la cabeza que invierta la preferencia.

In [9]:
def it_scores(S):
    t, a = get_attn(tok_bert, bert, S)
    i, an, st = t.index("it"), t.index("animal"), t.index("street")
    return a[:, :, i, an], a[:, :, i, st], t
an1, st1, t1 = it_scores(S1); an2, st2, t2 = it_scores(S2)
# queremos: animal>street en S1 (tired) y street>animal en S2 (wide)
score = (an1 - st1) - (an2 - st2)
top = torch.argsort(score.flatten(), descending=True)[:5]
best = [(int(i) // 12, int(i) % 12) for i in top]
for l, h in best:
    print(f"capa {l+1:2d} cabeza {h+1:2d}: tired  animal={an1[l,h]:.3f} street={st1[l,h]:.3f} | wide  animal={an2[l,h]:.3f} street={st2[l,h]:.3f}")
l, h = best[0]
_, a1 = get_attn(tok_bert, bert, S1); _, a2 = get_attn(tok_bert, bert, S2)
fig, axs = plt.subplots(1, 2, figsize=(12, 5))
heat(axs[0], a1[l, h].numpy(), t1, f"BERT capa {l+1}, cabeza {h+1} — «...too tired»")
heat(axs[1], a2[l, h].numpy(), t2, f"BERT capa {l+1}, cabeza {h+1} — «...too wide»")
plt.tight_layout(); plt.savefig("figs/bert_it.png", dpi=170); plt.show()

capa  7 cabeza 11: tired  animal=0.284 street=0.067 | wide  animal=0.012 street=0.590
capa  9 cabeza 11: tired  animal=0.676 street=0.000 | wide  animal=0.309 street=0.259
capa 10 cabeza  9: tired  animal=0.454 street=0.003 | wide  animal=0.023 street=0.045
capa 10 cabeza  1: tired  animal=0.608 street=0.002 | wide  animal=0.255 street=0.074
capa 11 cabeza  9: tired  animal=0.564 street=0.001 | wide  animal=0.245 street=0.078


C:\Users\aeeh2\AppData\Local\Temp\ipykernel_14364\94035697.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig("figs/bert_it.png", dpi=170); plt.show()


### C.3 GPT-2: matriz triangular inferior y «attention sink»

In [10]:
toks_g, att_g = get_attn(tok_gpt2, gpt2, S1)
print(toks_g)
upper = torch.triu(att_g, diagonal=1).abs().max().item()
print("Máx. atención sobre posiciones futuras:", upper, "-> triangular inferior:", upper < 1e-6)

# proporción de atención al primer token por capa (excluyendo la fila 0, donde es trivialmente 1)
TEXT = ("The animal didn't cross the street because it was too tired. Transformers rely on attention to relate "
        "every token with every other token in a sequence, which allows training on long contexts in parallel.")
tg, ag = get_attn(tok_gpt2, gpt2, TEXT)
first = ag[:, :, 1:, 0].mean(dim=(1, 2))
print("Atención media al primer token por capa:", np.round(first.numpy(), 3))
print("Media sobre capas:", first.mean().item())

fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
ax[0].bar(range(1, 13), first.numpy()); ax[0].set_xlabel("capa"); ax[0].set_ylabel("atención media al 1er token"); ax[0].set_title("GPT-2: attention sink")
heat(ax[1], att_g[5, 0].numpy(), toks_g, "GPT-2 capa 6, cabeza 1 (triangular inferior)")
plt.tight_layout(); plt.savefig("figs/gpt2_sink.png", dpi=170); plt.show()

['The', 'Ġanimal', 'Ġdidn', "'t", 'Ġcross', 'Ġthe', 'Ġstreet', 'Ġbecause', 'Ġit', 'Ġwas', 'Ġtoo', 'Ġtired', '.']
Máx. atención sobre posiciones futuras: 0.0 -> triangular inferior: True
Atención media al primer token por capa: [0.12  0.25  0.262 0.469 0.502 0.658 0.644 0.741 0.658 0.735 0.731 0.578]
Media sobre capas: 0.5289472937583923


C:\Users\aeeh2\AppData\Local\Temp\ipykernel_14364\884474237.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig("figs/gpt2_sink.png", dpi=170); plt.show()


### C.4 Entropía promedio de la atención por capa

In [11]:
def row_entropy(a):
    # a: (capas, cabezas, n, n). Entropía de Shannon (nats) por fila.
    return -(a * torch.log(a.clamp_min(1e-12))).sum(-1)       # (capas, cabezas, n)
tb, ab = get_attn(tok_bert, bert, TEXT)
entb, entg = row_entropy(ab), row_entropy(ag)
eb, eg = entb.mean(dim=(1, 2)), entg.mean(dim=(1, 2))
# GPT-2: la fila i solo ve i+1 tokens, se normaliza por la entropía máxima log(i+1)
norm = torch.log(torch.arange(1, ag.shape[-1] + 1).float())
eg_norm = (entg[:, :, 1:] / norm[1:]).mean(dim=(1, 2))
eb_norm = (entb / math.log(ab.shape[-1])).mean(dim=(1, 2))
print("BERT  entropía:", np.round(eb.numpy(), 2)); print("GPT-2 entropía:", np.round(eg.numpy(), 2))
print("BERT  normalizada:", np.round(eb_norm.numpy(), 2)); print("GPT-2 normalizada:", np.round(eg_norm.numpy(), 2))
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
ax[0].plot(range(1, 13), eb, "o-", label="BERT"); ax[0].plot(range(1, 13), eg, "s-", label="GPT-2")
ax[0].set_xlabel("capa"); ax[0].set_ylabel("entropía (nats)"); ax[0].legend(); ax[0].set_title("Entropía media")
ax[1].plot(range(1, 13), eb_norm, "o-", label="BERT"); ax[1].plot(range(1, 13), eg_norm, "s-", label="GPT-2")
ax[1].set_xlabel("capa"); ax[1].set_ylabel("H / H_max"); ax[1].legend(); ax[1].set_title("Entropía normalizada")
plt.tight_layout(); plt.savefig("figs/entropia.png", dpi=170); plt.show()

BERT  entropía: [2.87 2.36 1.96 2.16 1.9  1.49 1.46 1.33 1.64 1.97 2.14 2.15]
GPT-2 entropía: [1.8  2.08 1.69 1.27 1.17 1.07 1.16 0.88 1.17 0.93 0.97 1.43]
BERT  normalizada: [0.77 0.64 0.53 0.58 0.51 0.4  0.39 0.36 0.44 0.53 0.58 0.58]
GPT-2 normalizada: [0.66 0.75 0.62 0.47 0.42 0.39 0.43 0.32 0.42 0.33 0.34 0.5 ]


C:\Users\aeeh2\AppData\Local\Temp\ipykernel_14364\4131329360.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig("figs/entropia.png", dpi=170); plt.show()


### C.5 Especialización de cabezas (soporte para la discusión)
Se cuantifica el patrón dominante de cada cabeza de BERT: atención a [CLS]/[SEP], al token anterior, al siguiente o a sí misma.

In [12]:
n = ab.shape[-1]
idx = torch.arange(n)
def frac(a, sel): return (a * sel).sum((-1, -2)) / n
prev = (idx[:, None] - 1 == idx[None, :]).float(); nxt = (idx[:, None] + 1 == idx[None, :]).float()
selfm = torch.eye(n); sp = torch.zeros(n, n); sp[:, 0] = 1; sp[:, -1] = 1
P = {"anterior": frac(ab, prev), "siguiente": frac(ab, nxt), "misma": frac(ab, selfm), "[CLS]/[SEP]": frac(ab, sp)}
for k, v in P.items():
    l, h = np.unravel_index(v.argmax(), v.shape)
    print(f"{k:12s}: max={v.max():.2f} en capa {l+1} cabeza {h+1}; media={v.mean():.2f}")
dominant = torch.stack(list(P.values())).max(0).values
print("Cabezas cuyo patron dominante cubre >90% de la atencion:", int((dominant > 0.9).sum()), "de 144")

anterior    : max=0.64 en capa 4 cabeza 6; media=0.06
siguiente   : max=0.95 en capa 3 cabeza 10; media=0.07
misma       : max=0.40 en capa 12 cabeza 9; media=0.06
[CLS]/[SEP] : max=0.85 en capa 8 cabeza 7; media=0.35
Cabezas cuyo patron dominante cubre >90% de la atencion: 2 de 144


C:\Users\aeeh2\AppData\Local\Temp\ipykernel_14364\2651751614.py:8: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  l, h = np.unravel_index(v.argmax(), v.shape)
